In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Financial Report QA — Questions Over Annual Reports & Earnings Transcripts

## 1. Project Overview

**Task:** Parse annual reports and earnings call transcripts, index their content, and answer financial questions with grounded, cited answers — with special attention to tables and numerical data.

**Approach:** Build sample financial documents → parse sections + tables → chunk with number-aware strategies → embed in ChromaDB → retrieve → generate answers with numerical precision.

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — local LLM for generation
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — local embeddings
- `ChromaDB` — vector store
- `LangChain` — text splitting, orchestration

> **No API keys required.** Everything runs locally with Ollama.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Parse** annual reports and earnings transcripts into structured sections |
| 2 | **Handle tables** — preserve tabular structure through the RAG pipeline |
| 3 | **Handle numbers** — understand why LLMs struggle with arithmetic and how to mitigate |
| 4 | **Chunk financial text** — strategies that keep numbers in context |
| 5 | **Retrieve** relevant sections for financial questions |
| 6 | **Generate grounded answers** with citations and numerical precision |
| 7 | **Evaluate** retrieval and answer quality for financial data |

## 3. Problem Statement

Financial analysts, investors, and researchers spend hours reading 100+ page annual reports and multi-hour earnings transcripts. Common questions:

- *"What was revenue growth year-over-year?"*
- *"What did the CEO say about margins?"*
- *"How much was spent on R&D?"*
- *"What are the key risk factors?"*

A RAG system can locate the right section and extract the answer — but **financial data has unique challenges** that generic RAG doesn't handle well.

## 4. Why Financial Documents Are Hard for RAG

| Challenge | Why It's Hard | Example |
|-----------|--------------|--------|
| **Tables** | Embeddings lose row/column structure | Revenue table becomes word soup |
| **Numbers** | LLMs hallucinate arithmetic | "Revenue grew 15%" when it was 12% |
| **Units** | Millions vs billions vs thousands | "$4.2" could mean $4.2M or $4.2B |
| **YoY comparisons** | Numbers need adjacent context | 2025 revenue is meaningless without 2024 |
| **Footnotes** | Key qualifiers are elsewhere | "*Excluding one-time charges" |
| **Jargon** | Financial terms have precise meanings | "EBITDA", "diluted EPS", "free cash flow" |

## 5. Pipeline Architecture

```
Financial Documents (annual reports, earnings transcripts)
         │
  ┌──────┴──────────────────────────────────────┐
  │  Stage 1: DOCUMENT PARSING                   │
  │  • Detect document type (report vs transcript)│
  │  • Split into sections by headings            │
  │  • Detect & preserve tables separately        │
  │  • Extract metadata (company, year, type)     │
  └──────────────────────────────────────────────┘
         │
  ┌──────┴──────────────────────────────────────┐
  │  Stage 2: NUMBER-AWARE CHUNKING              │
  │  • Keep tables as single chunks              │
  │  • Larger chunks for financial text (600)    │
  │  • Preserve number + context together        │
  │  • Attach units/period metadata              │
  └──────────────────────────────────────────────┘
         │
  ┌──────┴──────────────────────────────────────┐
  │  Stage 3: EMBEDDING + INDEXING               │
  │  • all-MiniLM-L6-v2 (384-dim)              │
  │  • ChromaDB with rich metadata              │
  └──────────────────────────────────────────────┘
         │
  ┌──────┴──────────────────────────────────────┐
  │  Stage 4: RETRIEVAL + ANSWER GENERATION      │
  │  • Semantic search with metadata filters     │
  │  • Financial-specific system prompt          │
  │  • "Quote exact numbers" instruction         │
  │  • Confidence + limitations in output        │
  └──────────────────────────────────────────────┘
```

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface langchain-text-splitters chromadb sentence-transformers

print("Dependencies: langchain, langchain-ollama, chromadb, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports

In [ ]:
import os
import re
import json
import textwrap
import warnings
from pathlib import Path
from typing import Optional
from collections import Counter

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb

print("All imports OK")

## 8. Configuration

In [ ]:
LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
CHUNK_SIZE = 600          # Larger chunks for financial text
CHUNK_OVERLAP = 120       # More overlap to preserve number context
TABLE_CHUNK_SIZE = 1200   # Tables get bigger chunks to stay intact
TOP_K = 5
TEMP_ANSWER = 0.1         # Low temp for precise financial answers
TEMP_JUDGE = 0.0          # Zero temp for evaluation

print("Configuration")
print(f"  LLM: {LLM_MODEL}")
print(f"  Embeddings: {EMBED_MODEL}")
print(f"  Text chunks: {CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap")
print(f"  Table chunks: {TABLE_CHUNK_SIZE} chars (kept intact when possible)")
print(f"  Retrieval: top-{TOP_K}")

## 9. Helper Functions

In [ ]:
def clean(text: str) -> str:
    """Strip thinking tags from LLM output."""
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    """Extract JSON from LLM output."""
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    """Send a prompt to the LLM and return cleaned text."""
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    resp = llm.invoke(messages)
    return clean(resp.content)


def wrap_print(text: str, width: int = 90):
    """Print text with word wrapping."""
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM test
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Sample Financial Documents

## 10. Annual Report Excerpt

A simplified annual report with narrative sections AND financial tables. In production you'd parse real PDFs — here we define realistic text directly.

In [ ]:
ANNUAL_REPORT = {
    "filename": "nextera_annual_report_2025.md",
    "company": "NexTera Technologies",
    "year": "2025",
    "doc_type": "annual_report",
    "content": """# NexTera Technologies — Annual Report FY2025

## Company Overview
NexTera Technologies, Inc. is a global provider of enterprise cloud infrastructure
and AI-powered analytics solutions. Founded in 2011, the company serves over
12,000 enterprise customers across 45 countries. Headquarters: Austin, Texas.
Employees: 8,400 (up from 7,200 in FY2024).

## Financial Highlights

### Revenue Summary
Total revenue for FY2025 was $4.28 billion, representing a 22% increase from
$3.51 billion in FY2024. Revenue growth was primarily driven by cloud
infrastructure (up 31%) and the AI analytics segment (up 48%).

TABLE: Revenue by Segment ($ millions)
| Segment               | FY2025  | FY2024  | Change  |
|----------------------|---------|---------|--------|
| Cloud Infrastructure  | $2,140  | $1,634  | +31.0%  |
| AI Analytics          | $1,020  | $689    | +48.0%  |
| Professional Services | $680    | $712    | -4.5%   |
| Licensing & Other     | $440    | $475    | -7.4%   |
| **Total Revenue**     | **$4,280** | **$3,510** | **+21.9%** |

### Profitability
Gross margin improved to 68.2% from 65.8% in FY2024, driven by economies of
scale in cloud infrastructure and higher-margin AI analytics deals. Operating
income was $856 million (20.0% operating margin), up from $667 million (19.0%)
in FY2024.

TABLE: Profitability Metrics
| Metric                | FY2025    | FY2024    |
|----------------------|-----------|----------|
| Gross Margin          | 68.2%     | 65.8%    |
| Operating Income      | $856M     | $667M    |
| Operating Margin      | 20.0%     | 19.0%    |
| Net Income            | $642M     | $491M    |
| Diluted EPS           | $4.82     | $3.71    |
| Free Cash Flow        | $780M     | $598M    |

### Research & Development
R&D expenditure was $612 million in FY2025 (14.3% of revenue), compared to
$498 million in FY2024 (14.2% of revenue). Key investment areas included
large language model fine-tuning infrastructure, edge computing for IoT, and
quantum-resistant encryption. The company filed 142 new patents in FY2025.

## Risk Factors

### Competition
The cloud infrastructure market is highly competitive. Major competitors include
AWS, Microsoft Azure, and Google Cloud, each with significantly greater resources.
Our AI analytics segment faces competition from both established players (Palantir,
Databricks) and well-funded startups. Price pressure in cloud services could
compress margins in future periods.

### Regulatory Risk
Evolving AI regulations in the EU (AI Act), US state-level privacy laws, and
pending data sovereignty requirements in India and Brazil could increase compliance
costs. We estimate regulatory compliance costs could increase by $40-60 million
annually by FY2027.

### Customer Concentration
Our top 10 customers accounted for 28% of total revenue in FY2025, down from
32% in FY2024. No single customer exceeded 5% of revenue. Loss of a major
customer could materially impact results.

### Macroeconomic Conditions
Enterprise IT budgets remain sensitive to macroeconomic conditions. A prolonged
economic downturn could delay purchasing decisions, extend sales cycles, and
increase customer churn, particularly in the SMB segment which represents
approximately 15% of revenue.

## Outlook FY2026
Management expects total revenue of $5.1-5.3 billion for FY2026, representing
19-24% growth. Operating margin is expected to expand to 21-22%. Capital
expenditure is planned at $420-450 million, primarily for data center expansion
in Europe and Southeast Asia. The company plans to hire approximately 1,200
additional employees, primarily in engineering and sales.

TABLE: FY2026 Guidance
| Metric                | FY2026 Guidance       | FY2025 Actual |
|----------------------|----------------------|---------------|
| Revenue              | $5.1B - $5.3B        | $4.28B        |
| Revenue Growth       | 19% - 24%            | 21.9%         |
| Operating Margin     | 21% - 22%            | 20.0%         |
| CapEx                | $420M - $450M        | $385M         |
| Diluted EPS          | $5.60 - $5.90        | $4.82         |"""
}

print(f"Annual Report: {ANNUAL_REPORT['company']} FY{ANNUAL_REPORT['year']}")
print(f"  Length: {len(ANNUAL_REPORT['content']):,} chars")
num_tables = ANNUAL_REPORT['content'].count('TABLE:')
print(f"  Tables found: {num_tables}")

## 11. Earnings Call Transcript

In [ ]:
EARNINGS_TRANSCRIPT = {
    "filename": "nextera_q4_2025_earnings.md",
    "company": "NexTera Technologies",
    "year": "2025",
    "quarter": "Q4",
    "doc_type": "earnings_transcript",
    "content": """# NexTera Technologies — Q4 FY2025 Earnings Call Transcript
## Date: February 12, 2026
## Participants: Sarah Chen (CEO), Michael Torres (CFO), Analysts

### Opening Remarks — Sarah Chen, CEO

Thank you everyone for joining us today. I'm pleased to report that Q4 was our
strongest quarter ever, capping off a record fiscal year. Let me walk you through
the highlights.

Q4 revenue was $1.18 billion, up 24% year-over-year and 8% sequentially from Q3.
For the full year, we delivered $4.28 billion in revenue, exceeding the high end
of our initial guidance range of $4.0-4.2 billion.

Our AI Analytics segment continues to be the standout performer. Q4 AI revenue
was $298 million, up 55% year-over-year. We signed 84 new enterprise AI deals
in Q4, including 12 deals over $5 million in annual contract value. Our AI
pipeline entering FY2026 is the strongest I've seen in my tenure.

Cloud infrastructure also performed well, with Q4 revenue of $580 million, up
28% year-over-year. We added 340 net new customers in Q4, bringing our total
to over 12,000. Our dollar-based net retention rate was 118%, up from 114%
in Q4 FY2024.

### Financial Review — Michael Torres, CFO

Let me provide more detail on the financials.

Q4 gross margin was 69.1%, our highest ever, compared to 67.3% in Q4 FY2024.
The improvement reflects continued mix shift toward higher-margin AI analytics
and improved utilization in our cloud data centers. Full-year gross margin was
68.2%.

Q4 operating expenses were $578 million. R&D was $168 million (14.2% of revenue),
sales and marketing was $243 million (20.6%), and G&A was $167 million (14.1%).
We remain disciplined on hiring — headcount grew 4% sequentially in Q4.

Q4 operating income was $237 million, representing a 20.1% operating margin.
Net income was $178 million, or $1.34 per diluted share.

Free cash flow for Q4 was $215 million and $780 million for the full year. We
ended the year with $1.2 billion in cash and short-term investments and
$350 million in total debt.

TABLE: Q4 FY2025 Financial Summary
| Metric                | Q4 FY2025 | Q4 FY2024 | Change   |
|----------------------|-----------|-----------|----------|
| Revenue              | $1,180M   | $952M     | +24.0%   |
| Gross Margin         | 69.1%     | 67.3%     | +1.8 pts |
| Operating Income     | $237M     | $176M     | +34.7%   |
| Operating Margin     | 20.1%     | 18.5%     | +1.6 pts |
| Net Income           | $178M     | $131M     | +35.9%   |
| Diluted EPS          | $1.34     | $0.99     | +35.4%   |
| Free Cash Flow       | $215M     | $162M     | +32.7%   |

### Q&A Session

**Analyst (Morgan Stanley): Can you break down the AI analytics pipeline? What
percentage is from existing customers vs new logos?**

Sarah Chen: Great question. Roughly 60% of our AI pipeline is expansion within
existing customers — they start with a pilot, see results, and then expand to
more use cases. The remaining 40% is net new logos. We're particularly excited
about the financial services vertical, where we signed 15 new enterprise deals
in Q4 alone. The average deal size in financial services is approximately
$3.2 million annually, which is 40% higher than our overall average.

**Analyst (Goldman Sachs): What's driving the improvement in net retention rate
to 118%? And where do you think it can go?**

Michael Torres: The improvement from 114% to 118% is primarily driven by
customers adding AI analytics on top of their existing cloud infrastructure
deals. This cross-sell motion is very powerful — once a customer has their data
in our cloud, adding AI analytics is a natural next step. We believe net
retention can reach 120-122% over the next 12-18 months as AI attach rates
increase.

**Analyst (JP Morgan): You mentioned regulatory costs of $40-60M by FY2027.
What's the biggest driver there?**

Sarah Chen: The EU AI Act is the largest component — we estimate $20-25 million
annually for compliance, including mandatory audits, documentation, and
monitoring systems. US state privacy laws collectively add another $10-15 million.
Data sovereignty requirements in India and Brazil are the remaining $10-20
million, primarily in infrastructure costs to keep data in-country.

**Analyst (Barclays): Can you talk about the competitive environment in cloud
infrastructure? Are you seeing pricing pressure?**

Sarah Chen: Pricing is competitive but rational. We haven't seen the kind of
aggressive discounting that characterized the market in 2022-2023. Our strategy
is to compete on value, not price — our integrated cloud + AI platform gives us
a differentiation that pure cloud providers can't easily replicate. That said,
we are investing in our go-to-market motion in EMEA where AWS and Azure are
particularly strong. We plan to open 2 new data centers in Europe in FY2026.

**Analyst (UBS): What's the CapEx outlook and how should we think about
depreciation in FY2026?**

Michael Torres: We're guiding CapEx of $420-450 million for FY2026, up from
$385 million in FY2025. The incremental spend is roughly split 60/40 between
new data center capacity and AI training infrastructure. Depreciation will
increase to approximately $280-290 million in FY2026 from $245 million in
FY2025. The useful life assumption for our GPU clusters is 4 years."""
}

print(f"Earnings Transcript: {EARNINGS_TRANSCRIPT['company']} "
      f"{EARNINGS_TRANSCRIPT['quarter']} FY{EARNINGS_TRANSCRIPT['year']}")
print(f"  Length: {len(EARNINGS_TRANSCRIPT['content']):,} chars")
num_tables = EARNINGS_TRANSCRIPT['content'].count('TABLE:')
num_qa = EARNINGS_TRANSCRIPT['content'].count('**Analyst')
print(f"  Tables: {num_tables}, Analyst Q&A exchanges: {num_qa}")

---

# Part B — Document Parsing

## 12. Why Parsing Strategy Matters for Financial Documents

Generic text splitters treat everything as a flat string. Financial documents have **structure** that carries meaning:

1. **Section headings** → tell you the *topic* (Revenue, Risk, Outlook)
2. **Tables** → contain the *numbers* with column headers that give them meaning
3. **Q&A exchanges** → each question-answer pair is a unit of meaning
4. **Speaker identity** → CEO commentary vs CFO numbers vs analyst questions

Our parser preserves all of this.

In [ ]:
def detect_tables(text: str) -> list[dict]:
    """Find tables in text and return them with their positions."""
    tables = []
    # Pattern: TABLE: header followed by markdown table lines
    pattern = r"(TABLE:[^\n]*\n(?:\|[^\n]+\n)+)"
    for match in re.finditer(pattern, text):
        table_text = match.group(1).strip()
        # Extract table title
        title_match = re.match(r"TABLE:\s*(.+)", table_text)
        title = title_match.group(1).strip() if title_match else "Untitled Table"
        tables.append({
            "text": table_text,
            "title": title,
            "start": match.start(),
            "end": match.end(),
        })
    return tables


def parse_qa_exchanges(text: str) -> list[dict]:
    """Extract Q&A pairs from an earnings transcript."""
    exchanges = []
    # Split on analyst questions
    parts = re.split(r"(?=\*\*Analyst \()", text)
    for part in parts:
        part = part.strip()
        if not part.startswith("**Analyst"):
            continue
        # Extract analyst firm and question
        q_match = re.match(r"\*\*Analyst \(([^)]+)\):\s*(.+?)\*\*", part, re.DOTALL)
        if q_match:
            firm = q_match.group(1)
            question = q_match.group(2).strip()
            # Answer is everything after the question
            answer = part[q_match.end():].strip()
            exchanges.append({
                "firm": firm,
                "question": question,
                "answer": answer,
                "full_text": part,
            })
    return exchanges


def parse_financial_doc(doc: dict) -> list[dict]:
    """Parse a financial document into structured sections."""
    content = doc["content"]
    base_meta = {
        "company": doc["company"],
        "year": doc["year"],
        "doc_type": doc["doc_type"],
        "filename": doc["filename"],
    }
    if "quarter" in doc:
        base_meta["quarter"] = doc["quarter"]

    sections = []

    # 1. Detect and extract tables
    tables = detect_tables(content)
    for t in tables:
        sections.append({
            "text": t["text"],
            "metadata": {
                **base_meta,
                "section_type": "table",
                "section_title": t["title"],
                "has_numbers": "true",
            },
        })

    # 2. Extract Q&A exchanges (for transcripts)
    if doc["doc_type"] == "earnings_transcript":
        exchanges = parse_qa_exchanges(content)
        for ex in exchanges:
            sections.append({
                "text": ex["full_text"],
                "metadata": {
                    **base_meta,
                    "section_type": "qa_exchange",
                    "section_title": f"Q&A — {ex['firm']}",
                    "analyst_firm": ex["firm"],
                    "has_numbers": "true" if re.search(r'\$[\d,.]+|\d+%', ex["full_text"]) else "false",
                },
            })

    # 3. Split remaining text by headings (## or ###)
    # Remove table text to avoid duplication
    narrative = content
    for t in sorted(tables, key=lambda x: -x["start"]):
        narrative = narrative[:t["start"]] + narrative[t["end"]:]

    # Also remove Q&A section to avoid duplication
    if doc["doc_type"] == "earnings_transcript":
        qa_start = narrative.find("### Q&A Session")
        if qa_start >= 0:
            narrative = narrative[:qa_start]

    heading_parts = re.split(r"(?=^#{2,3}\s)", narrative, flags=re.MULTILINE)
    for part in heading_parts:
        part = part.strip()
        if not part or len(part) < 30:
            continue
        heading_match = re.match(r"^#{2,3}\s+(.+)", part)
        heading = heading_match.group(1).strip() if heading_match else "Introduction"

        # Detect if section contains numbers
        has_nums = bool(re.search(r'\$[\d,.]+|\d+\.?\d*%|\d{1,3}(?:,\d{3})+', part))

        # Detect speaker for transcripts
        speaker = ""
        if doc["doc_type"] == "earnings_transcript":
            spk_match = re.search(r"—\s*([^,]+),", heading)
            if spk_match:
                speaker = spk_match.group(1).strip()

        sections.append({
            "text": part,
            "metadata": {
                **base_meta,
                "section_type": "narrative",
                "section_title": heading,
                "has_numbers": "true" if has_nums else "false",
                **(({"speaker": speaker}) if speaker else {}),
            },
        })

    return sections


# Parse both documents
report_sections = parse_financial_doc(ANNUAL_REPORT)
transcript_sections = parse_financial_doc(EARNINGS_TRANSCRIPT)
all_sections = report_sections + transcript_sections

print(f"Parsed {len(all_sections)} total sections")
print(f"  Annual report: {len(report_sections)} sections")
print(f"  Earnings transcript: {len(transcript_sections)} sections")

print(f"\nSection types:")
type_counts = Counter(s["metadata"]["section_type"] for s in all_sections)
for st, count in type_counts.items():
    print(f"  {st}: {count}")

print(f"\nSections with numbers: "
      f"{sum(1 for s in all_sections if s['metadata']['has_numbers'] == 'true')}")

print(f"\nAll sections:")
for s in all_sections:
    m = s["metadata"]
    nums = "📊" if m["has_numbers"] == "true" else "📝"
    print(f"  {nums} [{m['section_type']:>12}] {m['doc_type'][:8]:8s} → {m['section_title'][:45]}")

---

# Part C — Handling Tables & Numbers

## 13. The Table Problem in RAG

Tables are the **most important part** of financial documents, but they're the worst fit for standard RAG:

### What Goes Wrong

1. **Embeddings lose structure:** The text `"| Revenue | $4,280 | $3,510 | +21.9% |"` embeds differently than `"Revenue was $4.28 billion, up 21.9%"` even though they say the same thing.

2. **Chunking splits rows from headers:** If a table spans 500+ chars and gets split, the second chunk has numbers without column labels — they become meaningless.

3. **Multi-row relationships:** To answer "which segment grew fastest?" the model needs ALL rows of the revenue table together.

### Mitigation Strategies

| Strategy | How | Trade-off |
|----------|-----|-----------|
| **Keep tables intact** | Don't split tables across chunks | Larger chunks, more tokens per query |
| **Linearize tables** | Convert to natural language alongside the table | Duplicates information, more storage |
| **Table-as-metadata** | Store table text as chunk but also add linearized summary | Best retrieval, more processing |
| **Separate table index** | Index tables in a different collection with specialized prompts | Complex, but best performance |

In [ ]:
def linearize_table(table_text: str) -> str:
    """Convert a markdown table to a natural language summary.
    This helps embeddings understand tabular data."""
    lines = table_text.strip().split("\n")

    # Extract title
    title = ""
    table_lines = []
    for line in lines:
        if line.startswith("TABLE:"):
            title = line.replace("TABLE:", "").strip()
        elif "|" in line and "---" not in line:
            table_lines.append(line)

    if len(table_lines) < 2:
        return table_text

    # Parse header
    headers = [h.strip() for h in table_lines[0].split("|") if h.strip()]

    # Parse data rows
    sentences = []
    if title:
        sentences.append(f"{title}:")

    for row_line in table_lines[1:]:
        cells = [c.strip() for c in row_line.split("|") if c.strip()]
        if len(cells) >= 2:
            row_label = cells[0].replace("**", "")
            parts = []
            for j, cell in enumerate(cells[1:], 1):
                if j < len(headers):
                    parts.append(f"{headers[j]}: {cell.replace('**', '')}")
            sentence = f"{row_label} — {', '.join(parts)}."
            sentences.append(sentence)

    return "\n".join(sentences)


# Demo: linearize the revenue table
revenue_table = next(s for s in all_sections
                     if s["metadata"]["section_type"] == "table"
                     and "Revenue by Segment" in s["metadata"]["section_title"])

print("ORIGINAL TABLE:")
print(revenue_table["text"][:400])
print(f"\n{'─'*60}")
print("LINEARIZED:")
linearized = linearize_table(revenue_table["text"])
print(linearized)

## 14. Number-Aware Chunking

### Key Principle: Never Separate a Number From Its Context

The number `$4.28 billion` is meaningless without knowing:
- **What** it measures (revenue)
- **When** (FY2025)
- **Who** (NexTera Technologies)
- **Compared to what** ($3.51 billion in FY2024)

Our chunking strategy:
1. **Tables → keep intact** (single chunk, even if large)
2. **Tables → also add linearized version** (better for retrieval)
3. **Narrative with numbers → larger chunks** (600 chars vs typical 400)
4. **Q&A exchanges → keep as complete pairs** (question + answer together)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", ", ", " "],
)

all_chunks = []

for section in all_sections:
    text = section["text"]
    meta = section["metadata"]
    sec_type = meta["section_type"]

    if sec_type == "table":
        # Strategy 1: Keep original table as one chunk
        all_chunks.append({
            "text": text,
            "metadata": {**meta, "chunk_type": "table_original",
                         "chunk_index": 0, "chunk_total": 1},
        })
        # Strategy 2: Also add linearized version for better retrieval
        linearized = linearize_table(text)
        all_chunks.append({
            "text": linearized,
            "metadata": {**meta, "chunk_type": "table_linearized",
                         "chunk_index": 0, "chunk_total": 1},
        })

    elif sec_type == "qa_exchange":
        # Keep Q&A pairs together — don't split
        if len(text) <= TABLE_CHUNK_SIZE:
            all_chunks.append({
                "text": text,
                "metadata": {**meta, "chunk_type": "qa_pair",
                             "chunk_index": 0, "chunk_total": 1},
            })
        else:
            # Very long Q&A — split but with large overlap
            big_splitter = RecursiveCharacterTextSplitter(
                chunk_size=TABLE_CHUNK_SIZE, chunk_overlap=200)
            parts = big_splitter.split_text(text)
            for i, p in enumerate(parts):
                all_chunks.append({
                    "text": p,
                    "metadata": {**meta, "chunk_type": "qa_pair",
                                 "chunk_index": i, "chunk_total": len(parts)},
                })

    else:
        # Narrative text — standard chunking with larger size
        if len(text) <= CHUNK_SIZE:
            all_chunks.append({
                "text": text,
                "metadata": {**meta, "chunk_type": "narrative",
                             "chunk_index": 0, "chunk_total": 1},
            })
        else:
            parts = text_splitter.split_text(text)
            for i, p in enumerate(parts):
                all_chunks.append({
                    "text": p,
                    "metadata": {**meta, "chunk_type": "narrative",
                                 "chunk_index": i, "chunk_total": len(parts)},
                })

print(f"Total chunks: {len(all_chunks)}")
print(f"Avg chunk length: {sum(len(c['text']) for c in all_chunks) / len(all_chunks):.0f} chars")

ct_counts = Counter(c["metadata"]["chunk_type"] for c in all_chunks)
print(f"\nChunk types:")
for ct, count in ct_counts.items():
    avg_len = sum(len(c["text"]) for c in all_chunks if c["metadata"]["chunk_type"] == ct) // count
    print(f"  {ct:>20}: {count:>3} chunks (avg {avg_len:>4} chars)")

print(f"\nChunks with numbers: "
      f"{sum(1 for c in all_chunks if c['metadata']['has_numbers'] == 'true')}")

---

# Part D — Embedding, Indexing & Retrieval

## 15. Build the Vector Store

In [ ]:
# Initialize embedding model
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

# Initialize ChromaDB
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("financial_docs")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="financial_docs",
    metadata={"hnsw:space": "cosine"},
)

# Embed and index
texts = [c["text"] for c in all_chunks]
metadatas = [c["metadata"] for c in all_chunks]
ids = [f"chunk_{i}" for i in range(len(all_chunks))]

print("Embedding all chunks...")
all_embeds = []
batch_size = 32
for i in range(0, len(texts), batch_size):
    batch = texts[i : i + batch_size]
    all_embeds.extend(embeddings.embed_documents(batch))
    print(f"  Batch {i // batch_size + 1}/{(len(texts) - 1) // batch_size + 1} done")

collection.add(
    documents=texts, embeddings=all_embeds,
    metadatas=metadatas, ids=ids,
)
print(f"\n✅ Vector store built: {collection.count()} chunks indexed")

## 16. Retrieval Functions

In [ ]:
def retrieve(query: str, top_k: int = TOP_K,
             where: Optional[dict] = None) -> list[dict]:
    """Retrieve top-k chunks for a query."""
    query_embed = embeddings.embed_query(query)
    kwargs = {"query_embeddings": [query_embed], "n_results": top_k}
    if where:
        kwargs["where"] = where
    results = collection.query(**kwargs)

    chunks = []
    for i in range(len(results["documents"][0])):
        chunks.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
        })
    return chunks


def display_chunks(chunks: list[dict], max_text: int = 130):
    """Pretty-print retrieved chunks."""
    for i, c in enumerate(chunks, 1):
        m = c["metadata"]
        sim = 1 - c["distance"]
        nums = "📊" if m.get("has_numbers") == "true" else "📝"
        print(f"\n  [{i}] {nums} (sim={sim:.3f}) {m.get('chunk_type','')}: "
              f"{m['section_title'][:45]}")
        print(f"      Doc: {m['doc_type']} | Company: {m['company']} "
              f"FY{m['year']}")
        print(f"      {c['text'][:max_text]}...")


# Test retrieval
print("BASIC RETRIEVAL TEST")
print("=" * 60)

q = "What was total revenue for the year?"
print(f"Q: {q}")
chunks = retrieve(q, top_k=3)
display_chunks(chunks)

## 17. Table-Prioritized Retrieval

For numerical questions, we want to prefer chunks that contain tables — they have the precise numbers.

In [ ]:
def retrieve_with_table_boost(query: str, top_k: int = TOP_K) -> list[dict]:
    """
    Retrieve chunks, but boost table chunks to the top.
    Retrieves 2x top_k, then re-ranks: tables first, then by similarity.
    """
    chunks = retrieve(query, top_k=top_k * 2)

    # Separate tables and non-tables
    tables = [c for c in chunks if c["metadata"].get("chunk_type", "").startswith("table")]
    non_tables = [c for c in chunks if not c["metadata"].get("chunk_type", "").startswith("table")]

    # Tables first, then non-tables, respecting similarity within each group
    reranked = tables + non_tables
    return reranked[:top_k]


# Compare standard vs table-boosted retrieval
q = "What were the revenue numbers by business segment?"

print("STANDARD RETRIEVAL")
print("=" * 60)
standard = retrieve(q, top_k=3)
display_chunks(standard)

print(f"\n\nTABLE-BOOSTED RETRIEVAL")
print("=" * 60)
boosted = retrieve_with_table_boost(q, top_k=3)
display_chunks(boosted)

print(f"\n→ Note: table-boosted retrieval prioritizes chunks with actual numbers.")

## 18. Filtered Retrieval

In [ ]:
print("FILTERED RETRIEVAL EXAMPLES")
print("=" * 60)

# Only from earnings transcript
q = "What did the CEO say about AI growth?"
print(f"\nQ: {q}")
print("Filter: doc_type = 'earnings_transcript'")
chunks = retrieve(q, where={"doc_type": "earnings_transcript"})
display_chunks(chunks[:3])

# Only tables
q = "What is the operating margin?"
print(f"\n{'─'*60}")
print(f"Q: {q}")
print("Filter: section_type = 'table'")
chunks = retrieve(q, where={"section_type": "table"})
display_chunks(chunks[:3])

# Only Q&A exchanges
q = "competitive environment cloud pricing"
print(f"\n{'─'*60}")
print(f"Q: {q}")
print("Filter: section_type = 'qa_exchange'")
chunks = retrieve(q, where={"section_type": "qa_exchange"})
display_chunks(chunks[:3])

---

# Part E — Answer Generation

## 19. Financial QA System Prompt

The system prompt is critical for financial accuracy. Key rules:
1. **Quote exact numbers** — never round, approximate, or compute
2. **Include units** — always say "$4.28 billion", never just "4.28"
3. **Cite sources** — reference the document, section, and period
4. **Don't do arithmetic** — if the user asks for a derived metric, quote the source numbers and let the user compute
5. **Flag uncertainty** — if the number isn't directly stated, say so

In [ ]:
FIN_SYSTEM = """You are a financial document assistant that answers questions
using ONLY the provided source documents (annual reports and earnings transcripts).

CRITICAL RULES FOR FINANCIAL ACCURACY:
1. QUOTE EXACT NUMBERS from the sources. Never round, estimate, or approximate.
2. ALWAYS INCLUDE UNITS — say "$4.28 billion" not "4.28" or "$4.28".
3. CITE SOURCES as [Source: document — section].
4. DO NOT PERFORM ARITHMETIC. If the user asks for a derived number (like
   percentage change), quote the source numbers and state whether the answer
   is directly stated vs needs calculation.
5. SPECIFY THE TIME PERIOD for every number — FY2025, Q4 FY2025, etc.
6. DISTINGUISH between actual results and forward guidance.
7. If the sources don't contain the answer, say so. Never guess.
8. Flag any inconsistencies between sources."""

FIN_PROMPT = """Answer this financial question using the source documents below.

QUESTION: {question}

SOURCE DOCUMENTS:
{sources}

Return ONLY JSON:
{{
  "answer": "detailed answer with exact numbers, units, periods, and [Source: ...] citations",
  "numbers_cited": [{{"metric": "name", "value": "exact value with units", "period": "time period"}}],
  "sources_used": ["document — section"],
  "confidence": "high|medium|low",
  "confidence_reason": "why",
  "is_directly_stated": true,
  "requires_calculation": false,
  "caveats": ["important notes"]
}}"""


def format_sources(chunks: list[dict]) -> str:
    parts = []
    for i, c in enumerate(chunks, 1):
        m = c["metadata"]
        doc_label = f"{m['doc_type']} FY{m['year']}"
        if 'quarter' in m:
            doc_label = f"{m['doc_type']} {m['quarter']} FY{m['year']}"
        parts.append(
            f"[Source {i}: {m['company']} {doc_label} — {m['section_title']}]\n"
            f"Type: {m.get('chunk_type', 'text')}\n"
            f"{c['text']}\n"
        )
    return "\n".join(parts)


def ask_financial(question: str, where: Optional[dict] = None,
                  boost_tables: bool = True) -> dict:
    """Full RAG pipeline for financial questions."""
    if boost_tables:
        chunks = retrieve_with_table_boost(question)
    else:
        chunks = retrieve(question, where=where)

    sources_text = format_sources(chunks)
    resp = ask(
        FIN_PROMPT.format(question=question, sources=sources_text),
        system=FIN_SYSTEM,
        temperature=TEMP_ANSWER,
    )
    result = parse_json(resp)
    if result:
        result["chunks"] = chunks
    else:
        result = {"answer": resp, "confidence": "unknown", "chunks": chunks}
    return result


def display_answer(question: str, result: dict):
    """Pretty-print a financial Q&A result."""
    conf = result.get("confidence", "?")
    badge = {"high": "🟢", "medium": "🟡", "low": "🔴"}.get(conf, "⚪")

    print(f"\n{'═'*70}")
    print(f"  Q: {question}")
    print(f"  Confidence: {badge} {conf}")
    direct = result.get("is_directly_stated", None)
    calc = result.get("requires_calculation", None)
    if direct is not None:
        print(f"  Directly stated: {'✅' if direct else '❌'}")
    if calc:
        print(f"  ⚠️ Requires calculation — verify independently")
    print(f"{'═'*70}")
    print()
    wrap_print(result.get("answer", ""))

    # Numbers cited
    numbers = result.get("numbers_cited", [])
    if numbers:
        print(f"\n  Numbers cited:")
        for n in numbers:
            print(f"    📊 {n.get('metric','')}: {n.get('value','')} "
                  f"({n.get('period', '')})")

    # Caveats
    caveats = result.get("caveats", [])
    if caveats:
        print(f"\n  Caveats:")
        for c in caveats:
            print(f"    ⚠ {c}")

    print(f"\n  Sources: {', '.join(result.get('sources_used', [])[:3])}")


print("Financial QA pipeline defined")

## 20. Ask Financial Questions

In [ ]:
# Q1: Direct number lookup
q1 = "What was NexTera's total revenue in FY2025?"
r1 = ask_financial(q1)
display_answer(q1, r1)

In [ ]:
# Q2: Table-based question
q2 = "Which business segment grew the fastest and by how much?"
r2 = ask_financial(q2)
display_answer(q2, r2)

In [ ]:
# Q3: Earnings call Q&A recall
q3 = "What did management say about the net retention rate and where it's heading?"
r3 = ask_financial(q3)
display_answer(q3, r3)

In [ ]:
# Q4: Forward-looking / guidance
q4 = "What is the revenue guidance for FY2026?"
r4 = ask_financial(q4)
display_answer(q4, r4)

In [ ]:
# Q5: Risk factors
q5 = "What are the main regulatory risks and estimated costs?"
r5 = ask_financial(q5)
display_answer(q5, r5)

In [ ]:
# Q6: Out of scope — data not present
q6 = "What was the company's revenue in FY2023?"
r6 = ask_financial(q6)
display_answer(q6, r6)

print(f"\n  → {'✅' if r6.get('confidence') == 'low' else '⚠'} "
      f"Expected LOW confidence (FY2023 data not in our documents)")

## 21. Batch Financial Q&A

In [ ]:
batch_questions = [
    "What is the gross margin and how did it change?",
    "How much was spent on R&D and what percentage of revenue is that?",
    "What is the diluted EPS for FY2025?",
    "How many new customers were added in Q4?",
    "What is the CapEx plan for FY2026?",
    "What is the company's cash position?",
    "What did the Barclays analyst ask about?",
]

print("BATCH FINANCIAL Q&A")
print("=" * 70)

for q in batch_questions:
    result = ask_financial(q)
    conf = result.get("confidence", "?")
    badge = {"high": "🟢", "medium": "🟡", "low": "🔴"}.get(conf, "⚪")
    direct = "📋" if result.get("is_directly_stated") else "🔧"
    answer = result.get("answer", "")
    print(f"\n  {badge}{direct} Q: {q}")
    print(f"       A: {textwrap.shorten(answer, width=95, placeholder='...')}")

---

# Part F — The Numbers Problem

## 22. Why LLMs Struggle With Numbers

This section explains a critical limitation when using LLMs for financial analysis.

### The Core Issue
LLMs are **language models**, not calculators. They:
- Predict the next token based on patterns, not mathematical logic
- Can quote numbers they've seen, but can't reliably **compute** with them
- Sometimes hallucinate plausible-looking but wrong percentages or totals

### What LLMs Can Do Well ✅
- **Quote** numbers directly from text ("Revenue was $4.28B")
- **Compare** pre-stated comparisons ("up 22% from FY2024")
- **Locate** which section contains a specific metric
- **Summarize** qualitative discussion around numbers

### What LLMs Do Poorly ❌
- **Calculate** percentages, ratios, or growth rates not stated in the text
- **Aggregate** across rows or sections ("total of all segments")
- **Compare** numbers from different parts of the document
- **Verify** if numbers in text match numbers in tables

In [ ]:
# Demonstration: Ask a question that REQUIRES calculation
q_calc = "What is the Professional Services revenue as a percentage of total revenue in FY2025?"

result = ask_financial(q_calc)
display_answer(q_calc, result)

# The correct answer: $680M / $4,280M = 15.89%
actual_pct = 680 / 4280 * 100

print(f"\n{'─'*70}")
print(f"VERIFICATION")
print(f"  Source numbers: $680M (Professional Services) / $4,280M (Total)")
print(f"  Correct answer: {actual_pct:.2f}%")
print(f"  → Check if the LLM's answer matches. If not, this demonstrates")
print(f"    why you should do arithmetic outside the LLM.")
print(f"  → The 'requires_calculation' flag should be True for this question.")

## 23. Strategies for Numerical Accuracy

| Strategy | How It Works | Limitation |
|----------|-------------|------------|
| **Quote-only mode** | Instruct LLM to only quote numbers, never compute | Can't answer derived questions |
| **Extract + compute** | LLM extracts numbers, Python does the math | Needs structured extraction |
| **Flag calculations** | LLM flags when an answer requires computation | User must verify manually |
| **Confidence scoring** | LLM rates its own confidence per number | Unreliable for edge cases |
| **Cross-reference** | Retrieve from multiple sections and check consistency | Expensive, but catches errors |

In [ ]:
# Strategy: Extract numbers from LLM, compute in Python

EXTRACT_PROMPT = """Extract the specific numbers needed to answer this question.
Do NOT compute the answer yourself.

QUESTION: {question}

SOURCES:
{sources}

Return ONLY JSON:
{{"numbers": [
    {{"label": "what this number represents",
      "value": numeric_value_only,
      "unit": "$ millions | % | count | etc",
      "period": "time period",
      "source": "where found"}}
  ],
  "calculation_needed": "description of what math to do"
}}"""


def extract_and_compute(question: str) -> dict:
    """Extract numbers with LLM, compute answer with Python."""
    chunks = retrieve_with_table_boost(question)
    sources_text = format_sources(chunks)

    resp = ask(
        EXTRACT_PROMPT.format(question=question, sources=sources_text),
        system="Extract numbers precisely. Do not calculate.",
        temperature=0.0,
    )
    extracted = parse_json(resp)
    return {"extracted": extracted, "chunks": chunks, "raw": resp}


# Example: What percentage of revenue is Professional Services?
q = "What percentage of total revenue came from Professional Services in FY2025?"
result = extract_and_compute(q)

print("EXTRACT + COMPUTE STRATEGY")
print("=" * 60)
print(f"Q: {q}\n")

if result["extracted"]:
    print("LLM extracted:")
    numbers = result["extracted"].get("numbers", [])
    for n in numbers:
        print(f"  📊 {n.get('label','')}: {n.get('value','')} {n.get('unit','')} "
              f"({n.get('period','')})")

    print(f"\n  Calculation needed: {result['extracted'].get('calculation_needed', '')}")

    # Python computes the answer
    vals = {n["label"]: n["value"] for n in numbers if isinstance(n.get("value"), (int, float))}
    print(f"\n  Extracted values: {vals}")

    if len(vals) >= 2:
        values_list = list(vals.values())
        if values_list[1] > 0:
            computed = values_list[0] / values_list[1] * 100
            print(f"  Python-computed answer: {computed:.2f}%")
        print(f"\n  ✅ Python did the math — no LLM hallucination risk.")
else:
    print(f"  Extraction failed. Raw response:")
    print(f"  {result['raw'][:300]}")

---

# Part G — Evaluation

## 24. Retrieval Accuracy Test

In [ ]:
retrieval_tests = [
    {"query": "revenue by segment breakdown",
     "expected_title": "Revenue by Segment",
     "expected_type": "table"},
    {"query": "Q4 earnings per share",
     "expected_title": "Q4 FY2025 Financial Summary",
     "expected_type": "table"},
    {"query": "what are the risk factors",
     "expected_title": "Competition",
     "expected_doc": "annual_report"},
    {"query": "AI analytics pipeline new vs existing customers",
     "expected_title": "Q&A — Morgan Stanley",
     "expected_type": "qa_exchange"},
    {"query": "FY2026 revenue guidance",
     "expected_title": "FY2026 Guidance",
     "expected_type": "table"},
    {"query": "regulatory compliance costs breakdown",
     "expected_title": "Q&A — JP Morgan",
     "expected_type": "qa_exchange"},
]

print("RETRIEVAL ACCURACY")
print("=" * 60)

correct = 0
for test in retrieval_tests:
    chunks = retrieve_with_table_boost(test["query"], top_k=3)
    top = chunks[0]["metadata"]

    # Check if expected content is in top-3
    found = False
    for c in chunks:
        m = c["metadata"]
        title_match = test.get("expected_title", "") in m.get("section_title", "")
        type_match = test.get("expected_type", "") in m.get("section_type", m.get("chunk_type", ""))
        doc_match = test.get("expected_doc", "") in m.get("doc_type", "")
        if title_match or (type_match and doc_match):
            found = True
            break

    if found:
        correct += 1
    status = "✅" if found else "❌"
    print(f"\n  {status} Q: {test['query'][:50]}")
    print(f"      Expected: {test.get('expected_title', test.get('expected_doc', ''))}")
    print(f"      Top result: {top['section_title'][:45]} ({top.get('chunk_type', '')})")

n = len(retrieval_tests)
print(f"\n{'─'*60}")
print(f"Accuracy (in top-3): {correct}/{n} ({correct/n*100:.0f}%)")

## 25. Numerical Accuracy Evaluation

In [ ]:
# Test: Can the system accurately quote specific numbers?
number_tests = [
    {"question": "What was total revenue in FY2025?",
     "expected": ["$4.28 billion", "$4,280", "4.28B", "4,280"]},
    {"question": "What was the Q4 FY2025 diluted EPS?",
     "expected": ["$1.34", "1.34"]},
    {"question": "What is the FY2025 free cash flow?",
     "expected": ["$780", "780M", "$780M", "780 million"]},
    {"question": "How many employees does the company have?",
     "expected": ["8,400", "8400"]},
    {"question": "What was R&D spending in FY2025?",
     "expected": ["$612", "612M", "$612M", "612 million"]},
]

print("NUMERICAL ACCURACY")
print("=" * 60)

correct = 0
for test in number_tests:
    result = ask_financial(test["question"])
    answer = result.get("answer", "")

    # Check if any expected number appears in the answer
    found = any(exp in answer for exp in test["expected"])
    if found:
        correct += 1
    status = "✅" if found else "❌"

    print(f"\n  {status} Q: {test['question']}")
    print(f"      Expected any of: {test['expected'][:3]}")
    # Find the number in the answer
    nums_in_answer = re.findall(r'\$[\d,.]+[BMK]?\b|\d+[.,]\d+%?', answer[:200])
    print(f"      Numbers in answer: {nums_in_answer[:5]}")

n = len(number_tests)
print(f"\n{'─'*60}")
print(f"Numerical accuracy: {correct}/{n} ({correct/n*100:.0f}%)")

## 26. Table vs Linearized Retrieval Comparison

In [ ]:
# Compare retrieval quality: original table vs linearized table
test_queries = [
    "What was the AI Analytics revenue?",
    "Which segment had declining revenue?",
    "What is the operating margin trend?",
]

print("TABLE FORMAT COMPARISON — Original vs Linearized")
print("=" * 60)

import numpy as np

for q in test_queries:
    q_embed = embeddings.embed_query(q)

    # Get all table chunks
    results = collection.query(
        query_embeddings=[q_embed], n_results=10,
        where={"section_type": "table"},
    )

    print(f"\n  Q: {q}")
    for i in range(min(4, len(results["documents"][0]))):
        meta = results["metadatas"][0][i]
        sim = 1 - results["distances"][0][i]
        chunk_type = meta.get("chunk_type", "")
        fmt = "📊 original" if "original" in chunk_type else "📝 linearized"
        print(f"    [{fmt}] sim={sim:.3f} — {meta['section_title'][:40]}")

print(f"\n{'─'*60}")
print("→ Linearized tables often retrieve BETTER for natural language queries")
print("  because embeddings understand sentences better than pipe-delimited rows.")
print("→ But the original table is better for the LLM to READ and extract numbers.")
print("→ Best practice: retrieve via linearized, but include both in the prompt.")

---

## 27. Common Mistakes

| Mistake | Why It's Wrong | What to Do Instead |
|---------|---------------|-------------------|
| **Splitting tables across chunks** | Second chunk has numbers without column headers | Keep entire tables intact as single chunks |
| **Asking LLM to compute** | "What's the YoY growth?" leads to hallucinated arithmetic | Extract numbers → compute in Python |
| **Missing units** | "Revenue was 4.28" — is that millions? billions? | Always attach units and use `has_numbers` metadata |
| **Mixing actuals and guidance** | "Revenue is $5.1B" — that's guidance, not actual | System prompt distinguishes actuals vs forward guidance |
| **Ignoring Q&A context** | Analyst Q&A contains key management commentary | Parse Q&A exchanges as separate chunks |
| **Same chunk size for everything** | 400-char chunks split financial paragraphs mid-number | Use 600+ chars for narrative, keep tables intact |
| **Not linearizing tables** | Markdown tables embed poorly for retrieval | Add linearized versions alongside originals |
| **No cross-referencing** | Report says one number, transcript says another | Retrieve from both and flag inconsistencies |

## 28. Production Improvement Ideas

1. **PDF parsing** — use `pymupdf` or `unstructured` to extract text + tables from real annual reports (10-K filings)
2. **Table extraction** — use `camelot` or `tabula` for precise table detection in PDFs
3. **SEC EDGAR integration** — auto-download 10-K, 10-Q, 8-K filings via the EDGAR API
4. **Multi-year index** — index 3-5 years of reports for trend analysis
5. **Hybrid search** — combine semantic search with keyword search for ticker symbols and financial terms
6. **Computation layer** — extract structured numbers → Python/pandas → compute derived metrics
7. **Chart generation** — auto-generate revenue/margin charts from extracted data
8. **Earnings calendar** — track upcoming earnings dates and auto-ingest new transcripts

## 29. Exercises

### Exercise 1: Multi-Period Analysis

In [ ]:
# ── Exercise 1: Add FY2024 report and compare across years ────────
# 1. Create a FY2024 annual report with different numbers
# 2. Index both years in the same ChromaDB collection
# 3. Ask: "How did revenue change from FY2024 to FY2025?"
# 4. Verify the LLM quotes both year's numbers correctly

# Hint: Add year to the metadata filter:
# retrieve(query, where={"year": "2025"})
# retrieve(query, where={"year": "2024"})

print("Exercise 1: Index multiple years and compare.")
print("Key insight: use metadata filters to retrieve from specific periods.")

### Exercise 2: Extract Structured Financial Data

In [ ]:
# ── Exercise 2: Build a structured financial data extractor ────────
# For each document, extract ALL key metrics into a structured dict:

EXTRACT_ALL_PROMPT = """Extract all key financial metrics from this document section.

SOURCE:
{source}

Return ONLY JSON:
{{"metrics": [
    {{"name": "metric name",
      "value": numeric_value,
      "unit": "$ millions | % | count",
      "period": "FY2025 | Q4 FY2025",
      "category": "revenue | profitability | efficiency | growth"}}
]}}"""

# Extract from the profitability table
profit_table = next(s for s in all_sections
                    if "Profitability" in s["metadata"]["section_title"]
                    and s["metadata"]["section_type"] == "table")

resp = ask(
    EXTRACT_ALL_PROMPT.format(source=profit_table["text"]),
    temperature=0.0,
)
extracted = parse_json(resp)

print("STRUCTURED EXTRACTION — Profitability Table")
print("=" * 60)

if extracted:
    for m in extracted.get("metrics", []):
        print(f"  📊 {m.get('name', ''):25s} {str(m.get('value', '')):>10s} "
              f"{m.get('unit', ''):>12s}  ({m.get('period', '')})")
    print(f"\n  → These structured values can feed into pandas, charts, or dashboards.")
else:
    print(f"  Extraction failed: {resp[:200]}")

### Exercise 3: Cross-Document Consistency Check

In [ ]:
# ── Exercise 3: Check if annual report and transcript agree ────────

CONSISTENCY_PROMPT = """Compare these two source documents for the SAME metric.
Do they agree? Are there discrepancies?

QUESTION: {question}

SOURCE 1 (Annual Report):
{source1}

SOURCE 2 (Earnings Transcript):
{source2}

Return ONLY JSON:
{{"metric": "what's being compared",
  "report_value": "from annual report",
  "transcript_value": "from earnings transcript",
  "consistent": true,
  "note": "explanation of any differences"}}"""

q = "What was the full-year FY2025 gross margin?"

# Retrieve from each document
report_chunks = retrieve(q, where={"doc_type": "annual_report"}, top_k=2)
transcript_chunks = retrieve(q, where={"doc_type": "earnings_transcript"}, top_k=2)

if report_chunks and transcript_chunks:
    resp = ask(
        CONSISTENCY_PROMPT.format(
            question=q,
            source1=report_chunks[0]["text"][:400],
            source2=transcript_chunks[0]["text"][:400],
        ),
        temperature=0.0,
    )
    result = parse_json(resp)

    print("CROSS-DOCUMENT CONSISTENCY CHECK")
    print("=" * 60)
    print(f"Q: {q}\n")

    if result:
        consistent = result.get("consistent", None)
        status = "✅ Consistent" if consistent else "❌ Discrepancy"
        print(f"  {status}")
        print(f"  Annual Report: {result.get('report_value', '?')}")
        print(f"  Transcript:    {result.get('transcript_value', '?')}")
        print(f"  Note: {result.get('note', '')}")
    else:
        wrap_print(resp[:300])

### Mini Challenge

1. **Earnings surprise detector:** Compare FY2025 actual results against the guidance ranges that would have been given at the start of the year. Did the company beat, meet, or miss guidance? Build a structured comparison table.

2. **Management tone analyzer:** Analyze the CEO's opening remarks for sentiment shifts. Compare phrases like "we are pleased" vs "challenges remain." Score optimism on a 1-10 scale with supporting quotes.

3. **Obligation extractor:** From the annual report's outlook section, extract all concrete commitments (CapEx range, hiring plans, data centers) into a timeline with deadlines and dollar amounts.

4. **Analyst question predictor:** Given the annual report's risk factors and financial results, use the LLM to predict what questions analysts would ask. Then compare against the actual Q&A in the transcript.

## 30. Session Summary

In [ ]:
print("SESSION SUMMARY")
print("=" * 50)

print(f"\nDocuments indexed:")
print(f"  📄 Annual Report: NexTera Technologies FY2025")
print(f"  📄 Earnings Transcript: Q4 FY2025")
print(f"  Total sections: {len(all_sections)}")
print(f"  Total chunks: {len(all_chunks)}")

print(f"\nPipeline stages:")
print(f"  ✅ Stage 1: Document parsing — sections, tables, Q&A exchanges")
print(f"  ✅ Stage 2: Number-aware chunking — tables intact, linearized copies")
print(f"  ✅ Stage 3: Embedding + ChromaDB with rich metadata")
print(f"  ✅ Stage 4: Retrieval with table-boost re-ranking")
print(f"  ✅ Stage 5: Financial QA with number precision rules")

print(f"\nTable/number handling:")
print(f"  ✅ Tables kept intact as single chunks")
print(f"  ✅ Linearized table copies for better retrieval")
print(f"  ✅ Table-boosted retrieval re-ranking")
print(f"  ✅ Extract + compute pattern (LLM extracts, Python calculates)")
print(f"  ✅ Cross-document consistency checking")

print(f"\nEvaluations:")
print(f"  ✅ Retrieval accuracy (section + type matching)")
print(f"  ✅ Numerical accuracy (exact number quoting)")
print(f"  ✅ Table format comparison (original vs linearized)")
print(f"  ✅ Calculation requirement detection")
print(f"  ✅ Out-of-scope handling")

## 31. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Parse by structure** — financial documents have sections, tables, and Q&A that must be preserved separately |
| 2 | **Never split tables** — a table row without its column headers is meaningless |
| 3 | **Linearize tables** for retrieval — embeddings understand sentences better than pipe-delimited rows |
| 4 | **Keep both formats** — linearized for search, original for the LLM to read |
| 5 | **LLMs can't do math** — quote numbers from sources, but compute derived metrics in Python |
| 6 | **Attach units always** — $4.28 without "billion" is useless and dangerous |
| 7 | **Distinguish actuals vs guidance** — "Revenue is $5.1B" might be a forecast, not a fact |
| 8 | **Cross-reference** annual report tables against earnings call numbers to catch inconsistencies |
| 9 | **Table-boost retrieval** — re-rank results to prioritize chunks with actual numbers |
| 10 | **Flag calculation requirements** — let the user know when a number was derived vs directly quoted |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*